In [ ]:
%pip install sqlalchemy pymysql pandas openpyxl

In [9]:
from sqlalchemy import create_engine, inspect, text
import pandas as pd
from urllib.parse import quote_plus

HOST = "localhost"
PORT = 3306
USER = "root"
PASSWORD = "root"

# Encoder le mot de passe pour gérer les caractères spéciaux
password_encoded = quote_plus(PASSWORD)

engine_cif = create_engine(
    f"mysql+pymysql://{USER}:{password_encoded}@{HOST}:{PORT}/cif"
)

engine_crm = create_engine(
    f"mysql+pymysql://{USER}:{password_encoded}@{HOST}:{PORT}/crm"
)

In [15]:
def inventorier_base(engine, nom_source):
    inspecteur = inspect(engine)
    resultats = []

    for table in inspecteur.get_table_names():

        table_sql = f"`{table.replace('`', '``')}`"

        # Nombre de lignes
        with engine.connect() as conn:
            nb_lignes = conn.execute(
                text(f"SELECT COUNT(*) FROM {table_sql}")
            ).scalar()

        # Clé primaire
        colonnes_pk = inspecteur.get_pk_constraint(table)
        pk = colonnes_pk.get("constrained_columns") or []

        cle_primaire = (
            ", ".join(pk) if pk else "Aucune déclarée"
        )

        # Colonnes
        colonnes = inspecteur.get_columns(table)
        noms_colonnes = [c["name"] for c in colonnes]
        nb_colonnes = len(colonnes)

        colonnes_nullables = sum(
            1 for c in colonnes
            if c.get("nullable", True)
        )

        with engine.connect() as conn:

            # Nombre total de valeurs NULL
            nb_null = 0

            for colonne in noms_colonnes:
                colonne_sql = f"`{colonne.replace('`', '``')}`"

                nb_null += conn.execute(
                    text(
                        f"SELECT COUNT(*) FROM {table_sql} "
                        f"WHERE {colonne_sql} IS NULL"
                    )
                ).scalar()

            # Nombre de doublons de clé primaire
            if pk:
                pk_sql = ", ".join(
                    f"`{col.replace('`', '``')}`"
                    for col in pk
                )

                requete_doublons = text(f"""
                    SELECT COALESCE(SUM(nb - 1), 0)
                    FROM (
                        SELECT COUNT(*) AS nb
                        FROM {table_sql}
                        GROUP BY {pk_sql}
                        HAVING COUNT(*) > 1
                    ) AS doublons
                """)

                nb_doublons_pk = conn.execute(
                    requete_doublons
                ).scalar()

                doublons = str(nb_doublons_pk)

            else:
                doublons = "N/A"

        # Toutes les statistiques dans une seule colonne
        observation = (
            f"{nb_colonnes} colonnes ; "
            f"{colonnes_nullables} colonnes nullable(s) ; "
            f"{nb_null} valeurs NULL ; "
            f"{doublons} doublons de clé primaire"
        )

        resultats.append({
            "Source": nom_source,
            "Table": table,
            "Lignes": nb_lignes,
            "Clé primaire": cle_primaire,
            "Observation chiffrée": observation
        })

    return resultats


# Inventaire CIF et CRM
resultats = []
resultats.extend(inventorier_base(engine_cif, "CIF"))
resultats.extend(inventorier_base(engine_crm, "CRM"))

inventaire = pd.DataFrame(
    resultats,
    columns=[
        "Source",
        "Table",
        "Lignes",
        "Clé primaire",
        "Observation chiffrée"
    ]
)

display(inventaire)

,Source,Table,Lignes,Clé primaire,Observation chiffrée
0,CIF,clients,40000,column0,10 colonnes ; 9 colonnes nullable(s) ; 26486 v...
1,CIF,compte,45000,column0,4 colonnes ; 3 colonnes nullable(s) ; 0 valeur...
2,CIF,pays,250,code_pays,2 colonnes ; 0 colonnes nullable(s) ; 0 valeur...
3,CRM,client,100000,client_id,13 colonnes ; 12 colonnes nullable(s) ; 133918...
4,CRM,contrat,100000,contrat_id,9 colonnes ; 8 colonnes nullable(s) ; 7460 val...
5,CRM,sinistre,100000,sinistre_id,10 colonnes ; 9 colonnes nullable(s) ; 0 valeu...
6,CRM,type_contrat,12,code_type,3 colonnes ; 0 colonnes nullable(s) ; 0 valeur...


In [16]:
inventaire.to_excel("inventaire_sources.xlsx", index=False)